In [1]:
# imports

# db adapter
import psycopg2

# load data into the db tables
from src.db_helpers import (
    table_insertion
)

# import pipeline variables
from src.config import (
    TOKEN,
    DB_NAME,
    PASSWORD,
    headers,
    rename_users_map,
    rename_repos_map,
    rename_commits_map,
    rename_issues_map
)

# main pipeline and function to retreive additional data for existing users
from src.pipeline import (
    run_pipeline,
    get_repos_data
)

# retrieve most recent ingested user
from src.db_helpers import (
    retrieve_last_user_id
)

In [2]:
# get most recent user id to get data from after that point
most_recent_user_id = retrieve_last_user_id(DB_NAME, PASSWORD)

# ensure env vars exists
print("TOKEN exists:", TOKEN is not None)
print("DB_NAME exists:", DB_NAME is not None)
print("PASSWORD exists:", PASSWORD is not None)
print("headers exists:", headers is not None)
print("rename_users_map exists:", rename_users_map is not None)
print("rename_repos_map exists:", rename_repos_map is not None)
print("rename_commits_map exists:", rename_commits_map is not None)
print("rename_issues_map exists:", rename_issues_map is not None)
print("most_recent_user_id exists:", most_recent_user_id is not None)

# vars to pass in the pipeline
pipeline_config = {
    "headers": headers,
    "rename_users_map": rename_users_map,
    "rename_repos_map": rename_repos_map,
    "rename_commits_map": rename_commits_map,
    "rename_issues_map": rename_issues_map,
    "most_recent_user_id": most_recent_user_id
}

TOKEN exists: True
DB_NAME exists: True
PASSWORD exists: True
headers exists: True
rename_users_map exists: True
rename_repos_map exists: True
rename_commits_map exists: True
rename_issues_map exists: True
most_recent_user_id exists: True


In [3]:
# fetch data
all_users_data, all_repos_data, all_commits_data, all_issues_data = run_pipeline(**pipeline_config)

# data size
print("Number of users:", len(all_users_data))
print("Number of repos:",len(all_repos_data))
print("Number of commits:", len(all_commits_data))
print("Number of issues:", len(all_issues_data))

Number of users: 60
Number of repos: 530
Number of commits: 7643
Number of issues: 423


In [4]:
# connect to postgres DB
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# insert data into staging tables
table_insertion(conn, 'users', all_users_data)
table_insertion(conn, 'repos', all_repos_data)
table_insertion(conn, 'commits', all_commits_data)
table_insertion(conn, 'issues', all_issues_data)

# close connection
conn.close()

In [5]:
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# get additional repos/commits/issues for existing users
repos_data, commits_data, issues_data = get_repos_data(5, conn, **pipeline_config)

# insert data into db staging tables
table_insertion(conn, 'repos', repos_data)
table_insertion(conn, 'commits', commits_data)
table_insertion(conn, 'issues', issues_data)

# data size
print("Number of repos:",len(repos_data))
print("Number of commits:", len(commits_data))
print("Number of issues:", len(issues_data))

conn.close()

Number of repos: 244
Number of commits: 3659
Number of issues: 155
